In [30]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import re

In [31]:
metrics = {
    "jmh": "bal_acc",
    "VMs": "bal_acc",
    "AzFuncInvoke": "smape",
    "WSD_3k": "smape",
    "AIOps": "auprc",
    "WSD": "auprc"
}

tasks = {
    "jmh": "steady-state",
    "VMs": "steady-state",
    "AzFuncInvoke": "timeseries-forecasting",
    "WSD_3k": "timeseries-forecasting",
    "AIOps": "anomaly-detection",
    "WSD": "anomaly-detection"
}

In [32]:
feature_metrics_variants = []
for metrics_file in Path("results/eval/ablation-models").glob("reshaped_metrics*"):
    variant_id = metrics_file.stem.split("#")[-1]
    
    metrics_df = pd.read_csv(metrics_file, index_col=[0])
    dataset_name = variant_id.split("---")[0]
    mean_metric = metrics_df[(metrics_df["model"] == "Meta")][metrics[dataset_name]].median()
    if metrics[dataset_name] == "smape":
        mean_metric = -mean_metric
    feature_metrics_variants.append({
        "variant_id": variant_id,
        "params": variant_id.split("---")[-1],
        "dataset": dataset_name,
        "metric": metrics[dataset_name], 
        "mdl_algo": metrics_file.stem.split("---")[1],
        "variant_mean_metric": mean_metric
    })
feature_metrics_variants = pd.DataFrame(feature_metrics_variants)
feature_metrics_variants = feature_metrics_variants[feature_metrics_variants["mdl_algo"] != "random_forest"] # RF is baseline
feature_metrics_variants

,variant_id,params,dataset,metric,mdl_algo,variant_mean_metric
0,AIOps---knn,knn,AIOps,auprc,knn,0.715794
1,AzFuncInvoke---linear,linear,AzFuncInvoke,smape,linear,-27.975785
2,WSD_3k---knn,knn,WSD_3k,smape,knn,-6.001342
3,AzFuncInvoke---knn,knn,AzFuncInvoke,smape,knn,-25.701497
6,WSD_3k---decision_tree,decision_tree,WSD_3k,smape,decision_tree,-5.850454
7,AzFuncInvoke---decision_tree,decision_tree,AzFuncInvoke,smape,decision_tree,-27.756007
8,VMs---knn,knn,VMs,bal_acc,knn,0.981132
9,VMs---linear,linear,VMs,bal_acc,linear,0.980560
10,WSD---linear,linear,WSD,auprc,linear,0.155182
11,WSD---knn,knn,WSD,auprc,knn,0.194901


In [33]:
choosen_variants = []
for rankfile in Path("results/eval/ablation-models").glob("reshaped_metrics*"):
    if "random_forest" not in rankfile.stem:
        continue
    metrics_df = pd.read_csv(rankfile, index_col=[0])
    dataset_name = rankfile.stem.split("#")[-1].split("---")[0]
    mean_metric = metrics_df[(metrics_df["model"] == "Meta")][metrics[dataset_name]].median()
    if metrics[dataset_name] == "smape":
        mean_metric = -mean_metric
    choosen_variants.append({
        "dataset": dataset_name,
        "metric": metrics[dataset_name], 
        "original_metric": mean_metric
    })
choosen_variants = pd.DataFrame(choosen_variants)
choosen_variants

,dataset,metric,original_metric
0,jmh,bal_acc,0.965714
1,WSD,auprc,0.184727
2,AzFuncInvoke,smape,-25.645839
3,WSD_3k,smape,-5.759666
4,VMs,bal_acc,0.980292
5,AIOps,auprc,0.778509


In [39]:
def compute_wins_df(df, metric):
    pivoted_df = df.pivot(columns="model", values=metric)

    wins = []
    loss = []

    for model in pivoted_df.columns:
        if model == "Meta":
            continue

        model_vals = pivoted_df[model]
        x_vals = pivoted_df['Meta']

        if metric == 'smape':
            # Lower is better
            better_mask = x_vals < model_vals
            worse_mask = x_vals > model_vals
        else:
            # Higher is better
            better_mask = x_vals > model_vals
            worse_mask = x_vals < model_vals

        equal_mask = x_vals == model_vals

        # Calculate counts
        better, worse, equal, total = better_mask.sum(), worse_mask.sum(), equal_mask.sum(), len(pivoted_df)

        # Calculate mean differences
        better_mean_diff = (x_vals[better_mask] - model_vals[better_mask]).mean()
        worse_mean_diff = (x_vals[worse_mask] - model_vals[worse_mask]).mean()
        equal_mean_diff = (x_vals[equal_mask] - model_vals[equal_mask]).mean()

        wins.append(round(100 * better / total, 1))
        loss.append(round(100 * worse / total, 1))

    # print(pivoted_df.columns)

    # print([(w, l) for w, l in zip(wins, loss)])

    # return wins, loss
    return [round(a - b, 1) for a, b in zip(wins, loss)]

def compute_wins(variant, baseline, metric):
    variant_df = pd.read_csv(f"results/eval/ablation-models/reshaped_metrics#{variant}.csv", index_col=[0])
    baseline_df = pd.read_csv(f"results/eval/reshaped_metrics_{baseline}.csv", index_col=[0])

    wins_variant = compute_wins_df(variant_df, metric)
    wins_baseline = compute_wins_df(baseline_df, metric)

    return [round(a - b, 1) for a, b in zip(wins_variant, wins_baseline)]

def compute_wins_var(variant, metric):
    variant_df = pd.read_csv(f"results/eval/ablation-models/reshaped_metrics#{variant}.csv", index_col=[0])

    wins_variant = compute_wins_df(variant_df, metric)
    # print(variant)
    
    return np.array(wins_variant)

def compute_wins_bs(baseline, metric):
    baseline_df = pd.read_csv(f"results/eval/reshaped_metrics_{baseline}.csv", index_col=[0])

    wins_baseline = compute_wins_df(baseline_df, metric)
    
    return np.array(wins_baseline)

def format_algo(row):
    if row['mdl_algo'] == "linear":
        return 'Linear'
    if row['mdl_algo'] == "knn":
        return "KNN"
    if row['mdl_algo'] == "decision_tree":
        return "Decision Tree"
    return row['mdl_algo']

In [40]:
feature_metrics_variants

,variant_id,params,dataset,metric,mdl_algo,variant_mean_metric,task,scores,wins_mean_diff,always_best
0,AIOps---knn,knn,AIOps,auprc,knn,0.715794,anomaly-detection,"[58.7, 31.0, 0.0, 86.2, 3.4]",-20.740000,False
1,AzFuncInvoke---linear,linear,AzFuncInvoke,smape,linear,-27.975785,timeseries-forecasting,"[5.9, 16.0, 20.1, 18.1, 62.2, 8.3]",-9.133333,True
2,WSD_3k---knn,knn,WSD_3k,smape,knn,-6.001342,timeseries-forecasting,"[-3.3, 46.6, 50.9, 51.9, 47.6, 28.1]",-11.450000,False
3,AzFuncInvoke---knn,knn,AzFuncInvoke,smape,knn,-25.701497,timeseries-forecasting,"[0.7, 5.2, 15.7, 12.8, 58.3, -3.1]",-15.966667,False
6,WSD_3k---decision_tree,decision_tree,WSD_3k,smape,decision_tree,-5.850454,timeseries-forecasting,"[-4.3, 44.3, 46.2, 51.4, 43.3, 27.6]",-13.666667,False
7,AzFuncInvoke---decision_tree,decision_tree,AzFuncInvoke,smape,decision_tree,-27.756007,timeseries-forecasting,"[6.6, 9.8, 22.6, 17.3, 50.7, 0.7]",-12.950000,True
8,VMs---knn,knn,VMs,bal_acc,knn,0.981132,steady-state,"[41.7, 29.3, 8.7]",-0.733333,True
9,VMs---linear,linear,VMs,bal_acc,linear,0.980560,steady-state,"[34.4, 23.3, 3.8]",-6.800000,True
10,WSD---linear,linear,WSD,auprc,linear,0.155182,anomaly-detection,"[18.6, 3.1, 2.5, 15.5, -4.4]",-10.160000,False
11,WSD---knn,knn,WSD,auprc,knn,0.194901,anomaly-detection,"[29.8, 11.2, 13.6, 28.6, 3.1]",0.040000,True


In [41]:
# =============== ABLATION STUDY FOR FEATURES ======================

# Step 1: Get the best variant per (dataset, algo)

feature_metrics_variants['task'] = feature_metrics_variants['dataset'].apply(lambda x: tasks[x])

feature_metrics_variants['scores'] = feature_metrics_variants.apply(
    lambda row: compute_wins_var(variant=row['variant_id'], metric=row['metric']),
    axis=1
)

feature_metrics_variants['wins_mean_diff'] = feature_metrics_variants.apply(
    lambda row: np.mean(compute_wins_var(variant=row['variant_id'], metric=row['metric']) - compute_wins_bs(baseline=f"{row['task']}___{row['dataset']}", metric=row['metric'])),
    axis=1
)

feature_metrics_variants['always_best'] = feature_metrics_variants.apply(
    lambda row: np.all(compute_wins_var(variant=row['variant_id'], metric=row['metric']) > 0),
    axis=1
)

# best_variants = feature_metrics_variants.sort_values('wins_var', ascending=False).groupby(['dataset', 'fs_algo']).first().reset_index()
best_variants = feature_metrics_variants

features_merge = best_variants.merge(choosen_variants, on=["dataset", "metric"])

features_merge["custom_mdl_algo"] = features_merge.apply(lambda row: format_algo(row), axis = 1)

features_merge = features_merge[["task", "dataset", "custom_mdl_algo", "wins_mean_diff"]].sort_values(by=["task", "dataset", "custom_mdl_algo"])

features_merge = features_merge.rename(columns={
    'custom_mdl_algo': 'Algorithm',
    'metric': 'Metric',
    'wins_mean_diff': 'Delta Wins',
})

# # mean_best_variants

print("Maximize the metric")
# Perform the pivot
pivoted_df = features_merge.pivot_table(
    index='Algorithm',
    columns=['task', 'dataset'],  # group columns by Task and Dataset
    values='Delta Wins'
)

pivoted_df = pivoted_df.rename(columns={
    'AzFuncInvoke': 'Azure'
})

# Optional: sort columns for readability
pivoted_df = pivoted_df.sort_index(axis=1, level=[0,1])#.applymap(lambda x: '(-)' if pd.notnull(x) and abs(x) < 1 else x)

pivoted_df.to_latex("results/tex/ablation_models.tex", column_format="l|cc|cc|cc", escape=True, float_format="%.1f")

Maximize the metric


In [42]:
pivoted_df

task          anomaly-detection        steady-state            \
dataset                   AIOps    WSD          VMs       jmh   
Algorithm                                                       
Decision Tree            -22.08 -14.50    -4.633333 -6.633333   
KNN                      -20.74   0.04    -0.733333 -6.400000   
Linear                   -12.46 -10.16    -6.800000 -5.433333   

task          timeseries-forecasting             
dataset                        Azure     WSD_3k  
Algorithm                                        
Decision Tree             -12.950000 -13.666667  
KNN                       -15.966667 -11.450000  
Linear                     -9.133333  -7.483333

In [48]:
import subprocess
import os
import tempfile

import re

def add_arrows_to_latex(tex_path: str, out_tex_path: str) -> str:
    """Read a .tex table, append red down-arrows to negative numeric cells, save modified version."""
    with open(tex_path, "r") as f:
        table_body = f.read()

    table_body = re.sub(
        r"(-\d+\.\d+)",
        r"\1 {\\color{red}$\\downarrow$}",
        table_body
    )

    with open(out_tex_path, "w") as f:
        f.write(table_body)

    return table_body


tex_path = "results/tex/ablation_models.tex"
modified_tex_path = "results/tex/ablation_models_arrows.tex"
pdf_path = "results/tex/ablation_models.pdf"

table_body = add_arrows_to_latex(tex_path, modified_tex_path)

full_latex = r"""\documentclass{article}
\usepackage{booktabs}
\usepackage{geometry}
\usepackage{xcolor}
\geometry{margin=1in}
\begin{document}
\pagestyle{empty}
""" + table_body + r"""
\end{document}
"""

with tempfile.TemporaryDirectory() as tmpdir:
    full_tex_path = os.path.join(tmpdir, "table.tex")
    with open(full_tex_path, "w") as f:
        f.write(full_latex)

    result = subprocess.run(
        ["pdflatex", "-interaction=nonstopmode", "table.tex"],
        cwd=tmpdir,
        capture_output=True,
        text=True
    )

    compiled_pdf = os.path.join(tmpdir, "table.pdf")
    if os.path.exists(compiled_pdf):
        import shutil
        shutil.copy(compiled_pdf, pdf_path)
        print(f"PDF saved to: {pdf_path}")
        print(f"Modified .tex saved to: {modified_tex_path}")
    else:
        print("pdflatex failed. Output:")
        print(result.stdout[-2000:])
        print(result.stderr[-1000:])

PDF saved to: results/tex/ablation_models.pdf
Modified .tex saved to: results/tex/ablation_models_arrows.tex
